Pipeline Version 2 для IEEE Fraud Detection

Шаги:
1. Загрузка случайной подвыборки данных через sample()
2. EDA: статистики пропусков, распределение классов, уникальных категорий
3. Отбор признаков: удаление колонок с > X% NaN, выбор кодирования категорий по числу уникальных значений
4. Обучение "простых" классификаторов: линейный SVM (SGDClassifier), LogisticRegression
5. Тестирование трёх энсемблевых регрессоров: DecisionTreeRegressor, BaggingRegressor, XGBoost


Импорты и настройки

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor
import xgboost as xgb

# Уровень логирования
import warnings
warnings.filterwarnings('ignore')

# Параметры
SAMPLE_FRAC = 0.3  # доля данных для подвыборки
NAN_THRESH = 0.6   # порог для удаления колонок с пропусками
CAT_ONEHOT_THRESH = 10  # if n_unique <= threshold -> OHE, else frequency encoding
SEED = 42

 Функция загрузки данных с рандомной подвыборкой

In [2]:
def load_data(sample_frac=SAMPLE_FRAC, seed=SEED):
    # Загрузка полных файлов
    train_trans = pd.read_csv('drive-download-20250512T040309Z-1-001/train_transaction.csv')
    train_id    = pd.read_csv('drive-download-20250512T040309Z-1-001/train_identity.csv')
    test_trans  = pd.read_csv('drive-download-20250512T040309Z-1-001/test_transaction.csv')
    test_id     = pd.read_csv('drive-download-20250512T040309Z-1-001/test_identity.csv')

    # Merge
    train = pd.merge(train_trans, train_id, on='TransactionID', how='left')
    test  = pd.merge(test_trans, test_id, on='TransactionID', how='left')

    # Подвыборка train
    train = train.sample(frac=sample_frac, random_state=seed)
    
    return train, test

Функции EDA: подсчёт NaN, распределение классов, уникальных значений категориальных

In [3]:
def eda_report(df):
    # Общая форма и класс баланса
    print('Shape:', df.shape)
    print('Fraud ratio:', df['isFraud'].mean())

    # NaN по колонкам
    nan_ratio = df.isna().mean().sort_values(ascending=False)
    print('\nTop NaN columns:')
    print(nan_ratio.head(10))

    # Категориальные
    cat_cols = df.select_dtypes(include=['object']).columns
    nunique = df[cat_cols].nunique().sort_values()
    print('\nCategorical unique counts:')
    print(nunique.head(10))


Функция отбор признаков: удаление по NaN и кодирование

In [4]:
def preprocess_v2(df_train, df_test, nan_thresh=NAN_THRESH, cat_thresh=CAT_ONEHOT_THRESH):
    # Целевая переменная
    y = df_train['isFraud']
    df_train = df_train.drop(['isFraud'], axis=1)

    # Совпадающие колонки
    common = list(set(df_train.columns)&set(df_test.columns))
    df_train = df_train[common]
    df_test  = df_test[common]

    # Удаление колонок с пропусками выше порога
    nan_frac = df_train.isna().mean()
    drop_cols = nan_frac[nan_frac>nan_thresh].index.tolist()
    df_train = df_train.drop(drop_cols, axis=1)
    df_test  = df_test.drop(drop_cols, axis=1)

    # Обработка признаков
    cat_cols = df_train.select_dtypes(include=['object']).columns
    num_cols = df_train.select_dtypes(include=['int64','float64']).columns

    # Заполняем пропуски
    for c in cat_cols:
        df_train[c].fillna('MISSING', inplace=True)
        df_test[c].fillna('MISSING', inplace=True)
    for c in num_cols:
        med = df_train[c].median()
        df_train[c].fillna(med, inplace=True)
        df_test[c].fillna(med, inplace=True)

    # Кодирование
    # OHE или frequency
    freq_encodings = {}
    transformers = []
    ohe_cols = []
    df_train_enc = pd.DataFrame()
    df_test_enc = pd.DataFrame()
    
    for c in cat_cols:
        n_uniq = df_train[c].nunique()
        if n_uniq<=cat_thresh:
            # One-hot
            ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            arr_train = ohe.fit_transform(df_train[[c]])
            arr_test  = ohe.transform(df_test[[c]])
            cols = [f"{c}_{cat}" for cat in ohe.categories_[0]]
            df_train_enc[cols] = arr_train
            df_test_enc[cols]  = arr_test
        else:
            # frequency encoding
            freq = df_train[c].value_counts(normalize=True)
            df_train_enc[c+'_freq'] = df_train[c].map(freq)
            df_test_enc[c+'_freq']  = df_test[c].map(freq).fillna(0)

    # Добавляем числовые признаки и масштабируем
    scaler = StandardScaler()
    df_train_enc[num_cols] = scaler.fit_transform(df_train[num_cols])
    df_test_enc[num_cols]  = scaler.transform(df_test[num_cols])

    # Заполняем возможные NaN после кодирования
    df_train_enc.fillna(0, inplace=True)
    df_test_enc.fillna(0, inplace=True)
    
    return df_train_enc, df_test_enc, y

Функция вывода метрик

In [5]:
def show_metrics(y_true, y_pred):
    acc = accuracy_score(y_true,y_pred)
    prec = precision_score(y_true,y_pred)
    rec = recall_score(y_true,y_pred)
    f1 = f1_score(y_true,y_pred)
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1: {f1:.4f}\n")

Основной блок: загрузка, EDA, предобработка, обучение моделей


In [6]:
 # Загрузка
train, test = load_data()
# EDA
eda_report(train)

# Предобработка v2
X, X_test, y = preprocess_v2(train, test)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)


Shape: (177162, 434)
Fraud ratio: 0.03579209988598006

Top NaN columns:
id_24    0.992182
id_25    0.991482
id_21    0.991471
id_08    0.991465
id_07    0.991465
id_26    0.991443
id_27    0.991437
id_23    0.991437
id_22    0.991437
dist2    0.936843
dtype: float64

Categorical unique counts:
M2       2
M3       2
M1       2
id_12    2
M8       2
M7       2
M6       2
M5       2
M9       2
id_36    2
dtype: int64


In [7]:
# Простые классификаторы
print("=== Linear SVM (SGDClassifier) ===")
svc = SGDClassifier(loss='hinge', random_state=SEED)
svc.fit(X_train, y_train)
show_metrics(y_val, svc.predict(X_val))

print("=== Logistic Regression ===")
lr = LogisticRegression(class_weight='balanced', solver='sag', max_iter=1000)
lr.fit(X_train, y_train)
show_metrics(y_val, lr.predict(X_val))


=== Linear SVM (SGDClassifier) ===
Accuracy: 0.9677
Precision: 0.7675
Recall: 0.1380
F1: 0.2340

=== Logistic Regression ===
Accuracy: 0.7734
Precision: 0.1109
Recall: 0.7603
F1: 0.1936



In [8]:
# Три регрессора
print("=== DecisionTreeRegressor ===")
dt = DecisionTreeRegressor(random_state=SEED)
dt.fit(X_train, y_train)
pred_dt = dt.predict(X_val) > 0.5
show_metrics(y_val, pred_dt)

print("=== BaggingRegressor ===")
bg = BaggingRegressor(random_state=SEED)
bg.fit(X_train, y_train)
pred_bg = bg.predict(X_val) > 0.5
show_metrics(y_val, pred_bg)

print("=== XGBoostRegressor ===")
xgr = xgb.XGBRegressor(use_label_encoder=False, eval_metric='logloss', random_state=SEED)
xgr.fit(X_train, y_train)
pred_xgb = xgr.predict(X_val) > 0.5
show_metrics(y_val, pred_xgb)

=== DecisionTreeRegressor ===
Accuracy: 0.9598
Precision: 0.4458
Recall: 0.5024
F1: 0.4724

=== BaggingRegressor ===
Accuracy: 0.9770
Precision: 0.8721
Recall: 0.4196
F1: 0.5666

=== XGBoostRegressor ===
Accuracy: 0.9771
Precision: 0.9057
Recall: 0.4014
F1: 0.5563

